# Ray Foundations, From One GPU to Many

## TLDR

You take an ordinary single-GPU PyTorch fine-tuning loop for gpt2 and turn it
into a synchronized four-GPU job with a handful of Ray calls. Ray Train wraps
the model for distributed data parallel, Ray Data streams and shards the
training data across workers, and a single report call records metrics and a
checkpoint. You write a normal training loop. Ray handles the distribution.
Going from 4 GPUs to 400 is one number in `ScalingConfig`.

## What is Ray?

<img src='https://docs.ray.io/en/releases-2.38.0/_images/map-of-ray.svg' width=700 />

## Introduction

In notebook 00 we argued that foundation models must scale. This notebook is the
first step in actually doing it. We start from the loop you already know, a
single GPU training a model on a stream of data, and we make three small changes
that hand the work to Ray.

The workload here is deliberately ordinary. We fine-tune gpt2, a 124 million
parameter language model, on a slice of the AG News text dataset. gpt2 fits
comfortably on one T4, which is the point. This notebook is about the
orchestration, not about fitting a giant model. That comes in notebook 02. Here
the lesson is that the same loop runs on one GPU or four with almost no change,
because Ray owns the distribution.

Two Ray libraries do the work. Ray Train launches and coordinates the worker
processes and wraps your model for distributed data parallel (DDP). Ray Data loads,
tokenizes, and shards the dataset so each worker sees its own slice. They plug
together through one call, `get_dataset_shard`.


## Key concepts used in this notebook

**`TorchTrainer`.** The Ray Train entry point. You give it a
`train_loop_per_worker` function and a `ScalingConfig`, and it launches one
worker process per GPU, each running your loop.

**`prepare_model`.** Wraps your `nn.Module` in PyTorch DistributedDataParallel
and moves it to the worker's GPU. It replaces manual `init_process_group` and
`DistributedDataParallel` calls.

**`get_dataset_shard`.** Inside the loop, hands this worker its slice of a Ray
Dataset that you passed to the trainer. It replaces `DistributedSampler` and
manual sharding.

**`ray.train.report`.** Records metrics from every worker and persists a
checkpoint from rank 0 to shared storage. It is also a synchronization barrier,
so every worker must call it.

**`ScalingConfig`.** One object that says how many workers to run and whether
they use GPUs. This is the knob you turn to scale.

**Ray Data `ActorPoolStrategy`.** Runs a stateful transform, here tokenization,
on a pool of actors that each load the tokenizer once and reuse it across
batches.


## What you will learn

- The controller and worker model behind Ray Train
- The three changes that turn a single-GPU loop into a distributed one
- How to build a Ray Dataset and tokenize it with an actor pool
- How each worker pulls its own data shard with `get_dataset_shard`
- How to report metrics and a checkpoint, and read them back from the result
- Why scaling from 4 GPUs to 400 is a one line change, and why the same code
  runs whether your GPUs sit on one node or many

## How Ray Data Works

Ray Data features a **streaming execution engine** that processes data in a pipelined fashion across a heterogeneous cluster of CPUs and GPUs. You write everything in native Python — your existing functions and libraries (NumPy, PyTorch, etc.) work as-is — and Ray Data handles the distribution, avoiding full materialization in memory and keeping all hardware utilized.

To understand why this matters, consider the difference between traditional batch processing and streaming:

**Traditional batch processing** completes one stage fully before starting the next, leading to idle resources:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/cko-2025-q1/batch-processing.png" width="80%" alt="Traditional Batch Processing">

**Streaming pipelining** overlaps stages, keeping all hardware (CPUs and GPUs) busy simultaneously:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/cko-2025-q1/pipelining.png" width="80%" alt="Streaming Model Pipelining">

This is critical for GPU-heavy workloads: while the GPU runs inference on one batch, the CPU can preprocess the next batch.

## How Ray Train Works

At a high level, Ray Train uses a controller (trainer) process to coordinate a group of training worker processes. The [Ray Train overview](https://docs.ray.io/en/latest/train/overview.html) introduces the core concepts: training function, workers, scaling configuration, and trainer.
<div align="center"><img src="https://docs.ray.io/en/latest/_images/overview.png" width="80%" loading="lazy"></div>

## Why Ray Train and Ray Data

| Challenge | Without Ray | With Ray |
|---|---|---|
| Multi-GPU DDP | `init_process_group`, device placement, `DDP(...)` | `prepare_model(model)` |
| Data sharding | `DistributedSampler`, manual splits per rank | `get_dataset_shard("train")` |
| Preprocessing at scale | A separate pipeline, or it blocks the GPU | Ray Data streams it on a CPU pool |
| Checkpoint coordination | Rank-0 filesystem handling by hand | `ray.train.report(checkpoint=...)` |
| Scale 4 to 400 GPUs | Rewrite launch and data plumbing | `ScalingConfig(num_workers=400)` |
| One node or many | Different launch scripts | Same code, Ray places the workers |


## Architecture

```
                  TorchTrainer (controller)
                          |  launches and coordinates
      +-----------+-------+-------+-----------+
   worker 0    worker 1    worker 2    worker 3      each on 1 x T4
   gpt2 DDP    gpt2 DDP    gpt2 DDP    gpt2 DDP      prepare_model wraps DDP
      +-----------+--- all-reduce gradients ---+
                          ^
                          |  get_dataset_shard("train")  ->  4 shards
            Ray Data stream (tokenized AG News, CPU actor pool)
                          |  report(metrics, checkpoint) from rank 0
                          v
                  /mnt/cluster_storage  (checkpoint + results)
```

The controller starts four worker processes, one per GPU. Ray Data tokenizes the
text on a pool of CPU actors and streams shards to the workers. Each worker runs
the same loop on its shard, gradients are averaged across workers by DDP, and
rank 0 writes the checkpoint to shared cluster storage.

## How this scales on Anyscale

| | This notebook | Production |
|---|---|---|
| Workers | 4 T4 GPUs on one node | Hundreds of GPUs across many nodes |
| Model | gpt2 (124M), fits per GPU | Sharded across GPUs (notebook 02) |
| Data | A 2 percent slice of AG News | Terabytes streamed from object storage |
| Change needed | -- | `ScalingConfig(num_workers=N)` and a bigger dataset path |

The code in this notebook does not change as you scale. You change the config
and the data source. That is the whole pitch.


## Cell 1 — Connect to Ray

**What you do.** Connect to the running cluster and ship the standard runtime
environment, which sends our `common` package to every worker and sets the env
vars these T4s need.

**What to check.** Four GPUs are reported. 

**Why it matters.** Every worker process needs the same code and environment.
`build_runtime_env` centralizes that so the rest of the notebook stays clean.


In [ ]:
import os
os.environ["RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO"] = "0"  # quiet a harmless driver tip

import ray
from common import utils

if not ray.is_initialized():
    ray.init(address="auto", runtime_env=utils.build_runtime_env())

utils.print_cluster_resources()


## The single-GPU starting point

Here is the loop we are migrating, written for a single GPU. Load gpt2, make an
optimizer, pull batches, and step. Nothing distributed yet. We define it for
reference and will not run a full training, since the distributed version below
is the real subject.

The three changes to make it distributed are small.

1. Wrap the model with `prepare_model` instead of calling `.to(device)` and
   `DistributedDataParallel` yourself.
2. Pull data with `get_dataset_shard` instead of a local `DataLoader`.
3. Replace your print and torch.save with `ray.train.report`.

Everything else, the forward pass, the loss, the optimizer step, stays the same.


## Cell 2 — The single-GPU loop, for reference

**What you do.** Define a plain single-GPU training step. This cell only defines
a function, it does not launch training.

**What to check.** There is nothing Ray-specific here. This is the baseline we
improve on.

**Why it matters.** Seeing the before makes the three changes obvious. Compare
this with the distributed loop a few cells down.

<div align="center"><img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-ai-libraries/diagrams/single_gpu_pytorch_v3.png" width="90%" loading="lazy"></div>

In [ ]:
import torch
from transformers import AutoModelForCausalLM

# A plain single-GPU loop, shown for contrast. This cell only defines it.
def single_gpu_train_step(model_name="gpt2", lr=5e-5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)   # change 1 target
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    # for batch in local_dataloader:                                      # change 2 target
    #     ids = batch["input_ids"].to(device)
    #     loss = model(input_ids=ids, labels=ids).loss
    #     optimizer.zero_grad(); loss.backward(); optimizer.step()
    # print(loss.item()); torch.save(model.state_dict(), ...)             # change 3 target
    return model

print("single_gpu_train_step defined, not launched")


## Cell 3 — Build the Ray Dataset

**What you do.** Load a small slice of AG News as a Ray Dataset and tokenize it
with an actor pool. The `Tokenize` class loads the gpt2 tokenizer once per actor
and reuses it across batches.

**What to check.** The dataset builds lazily. Nothing tokenizes until a
downstream step pulls data.

**Why it matters.** This is the Ray Data half of the integration. The same
pattern scales from a small slice to terabytes or more, because Ray Data streams rather
than loading everything into memory.


In [ ]:
import ray.data
from datasets import load_dataset

SEQ_LEN = 128
MODEL_NAME = "gpt2"

# A Ray Dataset from a 2 percent slice of AG News. Lazy until consumed.
raw_ds = ray.data.from_huggingface(load_dataset("ag_news", split="train[:2%]"))

class Tokenize:
    # Stateful transform. Loads the tokenizer once per actor.
    def __init__(self, model_name, seq_len):
        from transformers import AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(model_name)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.seq_len = seq_len

    def __call__(self, batch):
        import numpy as np
        enc = self.tok(
            list(batch["text"]),
            padding="max_length", truncation=True, max_length=self.seq_len,
        )
        return {
            "input_ids": np.array(enc["input_ids"], dtype=np.int64),
            "attention_mask": np.array(enc["attention_mask"], dtype=np.int64),
        }

train_ds = raw_ds.map_batches(
    Tokenize,
    fn_constructor_kwargs={"model_name": MODEL_NAME, "seq_len": SEQ_LEN},
    batch_size=256,
    compute=ray.data.ActorPoolStrategy(size=2),
)
print(train_ds)


## Cell 4 — Inspect a tokenized batch

**What you do.** Pull a tiny batch to confirm the pipeline produces the arrays
the model expects.

**What to check.** Each field has shape (batch, 128). This triggers the lazy
pipeline for just a few rows.

**Why it matters.** Checking a small batch before a long run is the cheapest way
to catch a preprocessing bug.


In [ ]:
sample = train_ds.take_batch(4)
{k: v.shape for k, v in sample.items()}

Ray Train solves common challenges in scaling deep learning:
*   **Scale**: Move from single GPU to multiple GPUs/nodes with minimal code changes.
*   **Infrastructure**: Abstracts away cluster management and resource provisioning.
*   **Observability**: Provides built-in dashboards for monitoring metrics, logs, and resource usage.
*   **Reliability**: Features automatic fault tolerance to recover from worker or node failures.

To migrate our PyTorch code to Ray Train, we need to adapt the model preparation, data loading, and the training loop.

The goal is to scale the single-GPU setup to a distributed data-parallel architecture:
<div align="center"><img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-ai-libraries/diagrams/multi_gpu_pytorch_v4.png" width="90%" loading="lazy"></div>

## Cell 5 — The per-worker training loop

**What you do.** Define `train_loop_per_worker`, the function Ray Train runs on
every GPU. The three changes from the single-GPU loop are marked in comments.

**What to check.** Find the three Ray touchpoints. `prepare_model` wraps DDP,
`get_dataset_shard` gives this worker its data, and `ray.train.report` records
metrics and a rank-0 checkpoint. Everything else is ordinary PyTorch.

**Why it matters.** This is the heart of the notebook. A normal loop becomes a
synchronized four-GPU job with three calls.

In [ ]:
import tempfile
import ray.train
import ray.train.torch
from ray.train import Checkpoint

def train_loop_per_worker(config):
    import torch, os, tempfile
    from transformers import AutoModelForCausalLM
    import ray.train, ray.train.torch
    from ray.train import Checkpoint

    device = ray.train.torch.get_device()
    world_size = ray.train.get_context().get_world_size()
    world_rank = ray.train.get_context().get_world_rank()

    # Change 1. prepare_model wraps the model in DDP and moves it to this GPU.
    model = AutoModelForCausalLM.from_pretrained(config["model_name"])
    model = ray.train.torch.prepare_model(model)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"])

    # Change 2. get_dataset_shard gives this worker its slice of the data.
    shard = ray.train.get_dataset_shard("train")
    per_worker_batch_size = config["global_batch_size"] // world_size

    model.train()
    running_loss, num_steps = 0.0, 0
    for step, batch in enumerate(shard.iter_torch_batches(
        batch_size=per_worker_batch_size,
        dtypes={"input_ids": torch.long, "attention_mask": torch.long},
        device=device,   # Ray Data moves the batch to this worker's GPU
    )):
        ids, attn = batch["input_ids"], batch["attention_mask"]
        loss = model(input_ids=ids, attention_mask=attn, labels=ids).loss
        optimizer.zero_grad()
        loss.backward()          # DDP all-reduces gradients across workers here
        optimizer.step()
        running_loss += loss.item()
        num_steps += 1
        if num_steps >= config["max_steps"]:
            break

    avg_loss = running_loss / max(num_steps, 1)

    # Change 3. report records metrics and a rank-0 checkpoint. It is a barrier,
    # so all workers call it.
    with tempfile.TemporaryDirectory() as tmp_dir:
        checkpoint = None
        if world_rank == 0:
            torch.save(model.module.state_dict(), os.path.join(tmp_dir, "model.pt"))
            checkpoint = Checkpoint.from_directory(tmp_dir)
        ray.train.report({"loss": avg_loss, "steps": num_steps}, checkpoint=checkpoint)


### Reporting Metrics and Checkpoints

Ray Train uses `ray.train.report()` to log metrics and report checkpoints to the Ray Train driver.

*   **Metrics**: Dictionaries of values (e.g., loss, accuracy) passed to `report()` are logged. By default, Ray Train only reports metrics from the rank 0 worker.
*   **Checkpoints**: Model states saved to a directory and passed as a `ray.train.Checkpoint`.

**Key Behaviors**:
1.  **Synchronization**: `ray.train.report()` acts as a global barrier. All workers must call it to ensure training stays in sync.
2.  **Efficient Checkpointing**: To avoid redundant uploads in standard DDP, only the rank 0 worker should save the checkpoint to disk. Ray Train then automatically persists it to your configured storage.

The following diagram shows this checkpoint lifecycle:

<div align="center"><img src="https://docs.ray.io/en/latest/_images/checkpoint_lifecycle.png" width="95%" loading="lazy"></div>

## Cell 6 — Configure and launch

**What you do.** Set the scaling and run configuration, then launch with
`TorchTrainer`. We pass the Ray Dataset under the name `train`, which is what
`get_dataset_shard("train")` reads.

**What to check.** The controller requests four GPUs, starts four workers, and
each prints a loss as it trains. The run writes to `/mnt/cluster_storage` so all
nodes can reach it.

**Why it matters.** `ScalingConfig(num_workers=4, use_gpu=True)` is the one place
that decides scale. Change the 4 and nothing else moves.


In [ ]:
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer

scaling_config = ScalingConfig(num_workers=4, use_gpu=True)  # NOTE: elastic training can be enabled by setting num_workers=(min_workers, max_worker) 

run_config = RunConfig(
    storage_path="/mnt/cluster_storage/ray_summit_foundations",
    name="gpt2_ray_foundations",
)

trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config={
        "model_name": MODEL_NAME,
        "lr": 5e-5,
        "global_batch_size": 32,   # 8 per worker across 4 workers
        "max_steps": 20,           # smoke scale, set higher for real training
    },
    scaling_config=scaling_config,
    run_config=run_config,
    datasets={"train": train_ds},
)

result = trainer.fit()
print("metrics:", result.metrics)


## Cell 7 — Inspect the result

**What you do.** Read the metrics and the checkpoint from the `Result` object,
load the fine-tuned weights, and generate a few tokens to confirm the model
works.

**What to check.** `result.metrics` has the loss and step count, and
`result.checkpoint` points at the saved model under `/mnt/cluster_storage`.
Recall
 that metrics populate from the rank-0 report that carries
the checkpoint.

**Why it matters.** The result is the bridge to everything after training,
serving, evaluation, or resuming. The checkpoint on cluster storage is what a
later job or a teammate picks up.


In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("metrics:", result.metrics)
print("checkpoint path:", result.path)

with result.checkpoint.as_directory() as ckpt_dir:
    state_dict = torch.load(os.path.join(ckpt_dir, "model.pt"), map_location="cpu")

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.load_state_dict(state_dict)
model.eval()

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
prompt = "Breaking news, the markets today"
ids = tok(prompt, return_tensors="pt").input_ids
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=20, do_sample=False, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0], skip_special_tokens=True))


## Scaling, the one number change

To run this on more GPUs you change one argument.

```python
scaling_config = ScalingConfig(num_workers=400, use_gpu=True)
```

Ray requests the workers, places them across whatever nodes the cluster has, and
runs the same loop. The data layer scales too. Point Ray Data at a larger source
and it streams and shards without loading everything into memory.

You can also make the worker count a range for elastic training, which we cover
in notebook 04. The key idea is that placement is Ray's job. Whether your four
GPUs are on one node or split across two, the code above does not change.


## Conclusion

You turned a single-GPU loop into a four-GPU distributed job with three changes.
`prepare_model` for DDP, `get_dataset_shard` for data, and `ray.train.report`
for metrics and checkpoints, all launched by a `TorchTrainer` with a
`ScalingConfig`. Ray Data streamed and tokenized the text on a CPU actor pool
and fed each worker its own shard.

Ray primitives you used. `TorchTrainer`, `prepare_model`, `get_dataset_shard`,
`ray.train.report`, `ScalingConfig`, `RunConfig`, and Ray Data with
`from_huggingface`, `map_batches`, and `ActorPoolStrategy`.

The catch is that gpt2 fits on one GPU, so plain data parallel works. Larger
foundation models do not fit. In notebook 02 we keep this exact Ray surface and
add sharding, FSDP and DeepSpeed ZeRO, so the model, gradients, and optimizer
states are split across GPUs.
